# Physics Solution-like Train Design Notebook

- 목적: `tools/physics_solution/full_physics_solution.py train-design`을 노트북에서 실행
- 비교 지표: 마지막에 `dev_logloss`만 출력


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / 'tools' / 'physics_solution' / 'full_physics_solution.py').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'tools' / 'physics_solution' / 'full_physics_solution.py').exists():
    PROJECT_ROOT = cwd.parent
elif (cwd.parent.parent / 'tools' / 'physics_solution' / 'full_physics_solution.py').exists():
    PROJECT_ROOT = cwd.parent.parent
else:
    raise FileNotFoundError('Could not locate project root from current notebook cwd')
print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
# 실행 설정
DATA_ROOT = (PROJECT_ROOT / 'data').resolve()
OUT_DIR = (PROJECT_ROOT / 'outputs' / 'physics_solution_like_train_design_v1.0').resolve()
MOTION_CSV = (DATA_ROOT / 'motion_targets.csv').resolve()

BACKBONE = 'efficientnet_v2_s'
USE_PRETRAINED = True
IMAGE_SIZE = 320
BATCH_SIZE = 8
EPOCHS = 12
SEED = 42
NUM_WORKERS = 8

print('DATA_ROOT =', DATA_ROOT)
print('OUT_DIR =', OUT_DIR)
print('MOTION_CSV =', MOTION_CSV)


In [ ]:
# motion CSV가 없으면 먼저 생성
if not MOTION_CSV.exists():
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / 'tools' / 'physics_solution' / 'full_physics_solution.py'),
        'extract-motion',
        '--data-root', str(DATA_ROOT),
    ]
    print('RUN:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('motion csv exists:', MOTION_CSV)


In [ ]:
# solution-like train-design 실행
cmd = [
    sys.executable,
    str(PROJECT_ROOT / 'tools' / 'physics_solution' / 'full_physics_solution.py'),
    'train-design',
    '--data-root', str(DATA_ROOT),
    '--out-dir', str(OUT_DIR),
    '--motion-csv', str(MOTION_CSV),
    '--backbone', BACKBONE,
    '--image-size', str(IMAGE_SIZE),
    '--batch-size', str(BATCH_SIZE),
    '--num-workers', str(NUM_WORKERS),
    '--epochs', str(EPOCHS),
    '--seed', str(SEED),
]
if USE_PRETRAINED:
    cmd.append('--pretrained')

print('RUN:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# 최종 비교 지표: dev_logloss만 동일하게 출력
report_path = OUT_DIR / 'dev_logloss_report.json'
assert report_path.exists(), f'report not found: {report_path}'
report = json.loads(report_path.read_text(encoding='utf-8'))
dev_logloss = report.get('dev_logloss')
print(f'dev_logloss: {dev_logloss:.10f}' if isinstance(dev_logloss, (int, float)) else f'dev_logloss: {dev_logloss}')
